###Staging Directory

In [0]:
catalog_name = dbutils.widgets.get('catalog_name')
spark.sql(F'USE CATALOG {catalog_name}')

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;


In [0]:
customers_df = spark.read.csv(path='/Volumes/wns24082026/quickstart_schema/sandbox/datasets/e-commerce/staging/customers/')

products_df = spark.read.json(path='/Volumes/wns24082026/quickstart_schema/sandbox/datasets/e-commerce/staging/products/')

orders_df = spark.read.json(path='/Volumes/wns24082026/quickstart_schema/sandbox/datasets/e-commerce/staging/orders/')

###Bronze Layer

In [0]:
%sql
DROP TABLE IF EXISTS bronze.customers;
DROP TABLE IF EXISTS bronze.orders;
DROP TABLE IF EXISTS bronze.products;


In [0]:
customers_df.write.saveAsTable('bronze.customers')
products_df.write.saveAsTable('bronze.products')
orders_df.write.saveAsTable('bronze.orders')


###Silver Layer

In [0]:
from pyspark.sql.functions import col

spark.read.table("bronze.orders").filter(col("order_id").isNotNull()).withColumn(
    "total_price", col("qty") * col("price")
).write.saveAsTable("silver.orders", mode="OVERWRITE")

###Gold Layer

In [0]:
from pyspark.sql.functions import sum

spark.read.table("silver.orders").groupBy("item_id").agg(sum("total_price").alias("total_revenue")).write.saveAsTable("gold.revenue_by_product", mode="overwrite")